# c04 - non-sequential vs sequential ray tracing: noise / time / memory

What this measures, in one line: **how many rays a differentiable ray trace needs to hold
per-bin Monte-Carlo noise under 10 %, and what that costs in seconds and GPU bytes.**

Per-bin relative noise is `eps = 1 / sqrt(rays per lit bin)`, and rays per lit bin is
`k * R / N**2` for a receiver of `N x N` bins and `R` launched rays. So bin count is
**not a free axis** - the sweep raises `R` and raises `N` together to pin `eps` just under
10 %, then reports time and memory at each rung. That is what maps the practical limit.

Two arms:

| arm | tracer | coating |
|---|---|---|
| `seq_R00` | dO `Lensgroup.trace` | R = 0 (all a sequential tracer can do) |
| `nonseq_mc_R02` | `diffoptics.nonseq.trace_mc` | R = 0.2, Russian-roulette MC |

**The comparison is compound.** The two arms differ in *both* tracer and scene - dO has no
partial reflection, so `seq @ R=0.2` does not exist. Every ratio below mixes the cost of
non-sequentiality with the cost of branching. `--arm nonseq_mc_R00` splits it into two clean
ratios if you want them.

Everything runs in **fp64**. On a T4 that is the 1/32-rate path, so expect roughly laptop-CPU
speed - that is a *finding*, not a bug, and the `--device cpu` cell puts it in the CSV as data.

**Runtime:** Runtime > Change runtime type > **T4 GPU**. Budget ~1 h for a full sweep.
Every cell is safe to re-run verbatim after a disconnect - results are append-only and
completed runs are skipped.

In [ ]:
# --- environment -------------------------------------------------------------
!nvidia-smi

import torch, platform
print('torch      ', torch.__version__)
print('cuda avail ', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('device     ', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f'VRAM       {total/2**30:.1f} GiB total, {free/2**30:.1f} GiB free')
print('python     ', platform.python_version())
print()
print('NOTE fp64 runs at 1/32 the fp32 rate on T4/L4, 1/2 on A100/V100.')
print('     On a 1/32-rate part expect roughly CPU speed.')

## 0. Get the code onto the runtime

The notebook only *drives* `c04_bench.py`; it does not contain it. The runtime needs the
`diffoptics` package plus `examples/nonseq/{c02_R02.py, c04_bench.py}`. Run **A or B**, not both.

**A - git clone** (preferred: reproducible, records a commit hash in `results.json`).
**B - upload `c04_colab_bundle.zip`** (64 KB, in `examples/nonseq/`) - no fork or PAT needed,
but `git_head` in the metadata will read `unknown`.

In [ ]:
# --- OPTION A: clone the branch ---------------------------------------------
# Put a GitHub PAT in Colab Secrets under the name GH_PAT (key icon, left sidebar).
# Never paste a token into a cell - it gets saved inside the notebook.
from google.colab import userdata

REPO   = 'Aly-Abdel-Motaleb/DiffOptics-nonseq'   # private
BRANCH = 'bench'

import os, subprocess
if not os.path.isdir('/content/DiffOptics'):
    tok = userdata.get('GH_PAT')
    url = f'https://{tok}@github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    url, '/content/DiffOptics'], check=True)
    # The clone URL carries the token, and git writes it in cleartext into
    # .git/config.  Drop the remote immediately - a shallow clone never pushes.
    subprocess.run(['git', '-C', '/content/DiffOptics', 'remote', 'remove', 'origin'])
%cd /content/DiffOptics
!git log --oneline -1
!pip install -q matplotlib psutil

In [ ]:
# --- OPTION B: upload the bundle instead of cloning --------------------------
# Pick examples/nonseq/c04_colab_bundle.zip when the file chooser opens.
import os, zipfile
if not os.path.isdir('/content/DiffOptics'):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('/content')
%cd /content/DiffOptics
!ls diffoptics examples/nonseq
!pip install -q matplotlib psutil

In [ ]:
# --- outputs go to Drive, not the container ---------------------------------
# A disconnect then costs at most the single run that was in flight.
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/diffoptics_bench/c04_out'
import os; os.makedirs(OUT, exist_ok=True)
BENCH = '/content/DiffOptics/examples/nonseq/c04_bench.py'
print(OUT)

## 1. Smoke - under 60 s

Does every code path execute, and does the energy ledger close? Run this before committing
an hour of GPU. It writes into a *separate* directory so it never pollutes the real CSV.

In [ ]:
!python $BENCH --smoke --device cuda --out $OUT/../smoke

## 2. Calibrate `k`

One pilot trace per arm, splatted at N in {16 ... 512}. Two things to eyeball:

* `slope_f` must be **-2.00 +- 0.10** - count per bin falls as `N**-2`, or the geometry is wrong.
* `k_mc / k_seq` must be **0.64** - the `T1T2` path fraction at R = 0.2, since at R = 0 every
  captured ray transmits. A free cross-check on the whole path-fraction story.

In [ ]:
!python $BENCH --calibrate-only --device cuda --out $OUT

## 3. Sequential arm - 24 runs, ~4 min

Cheap arm first, so the CSV is never empty if the session dies.

The ladder is **24 rows**: 1e4 3e4 1e5 3e5 1e6 3e6 1e7 at 3 source seeds (21) plus 2e7, 3e7 and
1e8 at 1 seed (3). The cell prints a `completed rows per arm` tally at the end.

**Both arms use the same chunk policy (`--chunk-above` now defaults to 5e7)**, so every rung of both is traced the same way:
monolithic through 3e7, chunked only at 1e8. That is what makes the memory and time columns
comparable rung for rung. Do not change it on one arm and not the other - a chunked row's peak
memory measures one *chunk*, not `R`, so mixing the two silently compares different quantities.

**Each arm then hits its own wall, and that is the result.** At ~300 B/ray the sequential arm
reaches 3e7 (8.6 GB) and OOMs at 1e8; at ~515 B/ray the non-sequential arm should clear 2e7
(~10.3 GB) and OOM around 3e7 (~15.5 GB). The ratio of those two walls is the memory cost of
non-sequential tracing, stated in rays rather than in bytes.

When a monolithic run OOMs, the row is written with `status='oom'` and every larger monolithic
rung of that arm is skipped rather than retried - the wall is already found. Their memory comes
from the dotted extrapolation in `memory_vs_rays.png`.


In [ ]:
!python $BENCH --arm seq_R00 --device cuda --out $OUT


## 4. Non-sequential MC arm - 24 runs, same treatment

**Identical flags to the sequential cell**, so the comparison is head to head at every rung:
same ray counts, same seeds, same chunk policy, same bin rule. The only differences left are the
ones under study - the tracer and the scene.

Expect this arm to OOM earlier than the sequential one. That is not a failure of the run; it is
the measurement. The sweep records the wall and stops climbing.

**Re-run this cell verbatim after a disconnect.** Completed runs are keyed on
`(arm, device, rays, seed, rep, chunk, n_bins)` and skipped; only `ok` and `oom` rows count as
done, so anything that errored is retried automatically.

One asymmetry that no flag can remove, and the report must state: the sequential arm captures
every launched ray (`f_path = 1.0`) while the MC arm splits power across paths, so at equal `R`
the sequential arm legitimately earns more bins. That is why the headline plot is **N at fixed
10 % noise**, not noise at fixed N.


In [ ]:
!python $BENCH --arm nonseq_mc_R02 --device cuda --out $OUT


### 4b. Past the wall - chunked extension, both arms

The rungs above each arm's wall have no monolithic answer. Chunking gets the *noise* curve up to
1e8 anyway: peak memory is then set by `--chunk-size`, not by `R`, so those rows are excluded
from the memory fit but are perfectly good noise and time points.

Run for **both arms with the same flags**, so the extension is as symmetric as the ladder below
it. `chunk` is part of the resume key, so these rows sit alongside the monolithic ones rather
than replacing them - a rung can hold both, which is exactly what the wall measurement needs.


In [ ]:
!python $BENCH --arm seq_R00 --device cuda --chunk-above 1e7 --rays 2e7 3e7 1e8 --seeds 1 --reps 1 --out $OUT
!python $BENCH --arm nonseq_mc_R02 --device cuda --chunk-above 1e7 --rays 2e7 3e7 1e8 --seeds 1 --reps 1 --out $OUT


### Coverage check - did the full ladder actually land?

Reads `results.csv` directly, so it costs nothing and works after a reconnect without re-running
either arm. The ladder is **24** rungs per arm.

Read it by *rungs*, not by row count. The wall probe deliberately adds a second, monolithic row
at 2e7/3e7/1e8, so a healthy CSV can hold **more** than 24 rows per arm - and a rung whose
monolithic copy OOM'd is still answered as long as its chunked copy succeeded. Only
`rungs with no successful row` is a real hole; re-run that arm's cell above and it is retried.


In [ ]:
import pandas as pd

d = pd.read_csv(f'{OUT}/results.csv')
d = d[d['mode'] == 'adaptive']
LADDER = 24                       # 7 rungs x 3 seeds + 2e7 + 3e7 + 1e8 x 1 seed

for arm in ('seq_R00', 'nonseq_mc_R02'):
    a = d[d['arm'] == arm]
    ok = a[a['status'] == 'ok']
    # A rung is answered if ANY of its rows succeeded - the monolithic copy may
    # have OOM'd while the chunked copy of the same rung is fine, and that pair
    # is the wall measurement, not a hole in the ladder.
    print(f"{arm:15s} ok={len(ok):3d}/{LADDER}   "
          f"oom={int((a['status'] == 'oom').sum())}  "
          f"error={int((a['status'] == 'error').sum())}  "
          f"ledger_fail={int((a['status'] == 'ledger_fail').sum())}")
    if a.empty:
        print('    arm not run yet')
        continue
    missing = sorted(set(a['rays']) - set(ok['rays']))
    if missing:
        print('    rungs with no successful row:', [f'{m:.0e}' for m in missing])
    if len(ok) < LADDER and not missing:
        print('    every rung answered; the shortfall is duplicate rows '
              '(monolithic + chunked) or missing seeds, not missing rungs')


## 5. Optional extras

* **fixed-N slope check** - the sharpest single test in the harness: at fixed `N`, `eps` must
  fall as `R**-1/2`. Validates the noise measurement, the splat, the classifier and RNG
  independence at once. Gate: **-0.500 +- 0.02**, fitted on `eps_median` (the p10 statistic is
  the right headline for sizing bins but is itself biased at small integer counts).
* **CPU ladder** - the fp64 finding. Slow, so keep it to R <= 1e6.
* **`nonseq_mc_R00`** - decomposes the compound comparison.
* **`split_R02`** - deterministic branching tracer; expect 5-8x the bytes per ray, which is
  what turns "MC is the memory-scalable one" from assertion into measurement.

In [ ]:
!python $BENCH --slope --device cuda --seeds 2 --reps 1 --out $OUT
# !python $BENCH --device cpu --seeds 1 --reps 1 --rays 1e4 3e4 1e5 3e5 1e6 --out $OUT
# !python $BENCH --arm nonseq_mc_R00 --device cuda --out $OUT
# !python $BENCH --arm split_R02 --device cuda --rays 1e4 1e5 1e6 --out $OUT

## 6. Plots and summary

Four figures:

* `noise_vs_rays.png` - eps held at the 10 % target as N rises with R, plus the fixed-N
  `R**-1/2` panel if you ran the slope cell.
* `time_vs_rays.png` - the cost curve. Chunked rows carry their own markers.
* `memory_vs_rays.png` - **linear GB against evenly spaced rungs**, deliberately not log-log.
  A log y axis puts 2.9 GB and 5.1 GB a few millimetres apart and invites the reader to conclude
  the two tracers cost about the same; they do not, and the gap - one arm running out of memory a
  whole rung before the other - is the finding. Solid is measured (monolithic only), dotted is
  the linear `B/ray` extrapolation past the last real point, the dashed black `Limit` is the
  device VRAM, and a red x marks an observed OOM. A predicted crossing that lands near the
  observed OOM is what validates the linear-memory claim.
* `bin_count_hist.png` - the distribution behind the headline `eps_p10`, drawn at R=1e6 only.

Chunked rows are excluded from the memory figure entirely: their peak measures one *chunk*, not
`R`, so plotting them beside monolithic rows would claim a memory number that is not real.

If `bin_count_hist.png` is missing, the R=1e6 rung was already recorded before the histogram
existed. The count maps are dumped during the trace, so it needs that one run redone - the
cell below does exactly that and nothing else.


In [ ]:
!python $BENCH --plots-only --out $OUT
!python $BENCH --json-only  --out $OUT

from IPython.display import Image, display
import os
for f in ('noise_vs_rays.png', 'time_vs_rays.png', 'memory_vs_rays.png',
          'bin_count_hist.png'):
    if os.path.exists(f'{OUT}/{f}'):
        display(Image(filename=f'{OUT}/{f}'))
    else:
        print('missing:', f)

In [ ]:
# Only needed if bin_count_hist.png did not appear above.
# Redoes the single R=1e6 rung so the per-bin count maps get dumped; ~15 s.
!python $BENCH --rays 1e6 --seeds 1 --device cuda --force --out $OUT
!python $BENCH --plots-only --out $OUT

from IPython.display import Image, display
display(Image(filename=f'{OUT}/bin_count_hist.png'))

In [ ]:
import pandas as pd, json
df = pd.read_csv(f'{OUT}/results.csv')
print(df.status.value_counts().to_dict())

ok = df[(df.status == 'ok') & (df['mode'] == 'adaptive')]
cols = ['n_bins_fwd', 'n_bins_back', 'eps_p10_fwd', 'frac_above_fwd',
        't_wall_s', 'mem_alloc_mb']
display(ok.groupby(['arm', 'rays'])[cols].mean().round(4))

print(json.dumps(json.load(open(f'{OUT}/results.json'))['arms'], indent=2)[:2000])

## 7. Findings - fill these in from the run above

1. **Cost per ray.** `us_per_ray` and `bytes_per_ray` per arm, straight out of `results.json`.
   Quote the raw non-seq/seq ratio *and* a per-surface-interaction-normalised one: the raw
   number compares 2 fixed refractions against up to 10 closest-hit rounds over 2 elements,
   so the honest overhead lies between the two.
2. **The OOM wall.** Largest R that fitted, smallest that failed, and the wall predicted from
   the linear bytes-per-ray fit. Predicted ~ observed is what validates linearity - `trace_mc`
   is flat-`[N]`-width by construction, so bytes per ray must not depend on depth.
3. **Bins at 10 % noise.** `N` reached at each `R`, forward vs backward. The forward/backward
   gap at equal budget quantifies how much harder the reflected path is to resolve - a
   statement no sequential tracer can make at all.
4. **fp64 on a T4.** GPU vs CPU wall time at matched R. If they are close, say so plainly:
   the differentiable non-sequential tracer in fp64 runs no faster on a T4 than on a laptop CPU.
5. **The compound-comparison caveat**, restated in the report itself. The sequential arm also
   cannot produce `R1`, `T1R2T1` or the ghost at all - `n_bins_back` is NaN on those rows by
   construction, not by omission.
6. **The bin-count histogram** (`bin_count_hist.png`, R=1e6). `eps_p10` is one number off
   the tail of this distribution; the plot is what it summarises. Read the *bottom* row: the
   fraction of lit bins past the 10 % line is the requirement, stated directly. Note the
   spread in the top row is mostly the irradiance profile of the receiver, not shot noise -
   do not read it as a Poisson test.
7. **Floored rows.** Low-R rows marked `bin_flag='floored'` sit above 10 % because `N` hit its
   16-bin floor. They are kept deliberately: they anchor the `-1/2` slope check. Do not read
   them as failures.